In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ---------------------------------------------------------
# 1. SETTINGS
# ---------------------------------------------------------

np.random.seed(7)

# Initial 2D data point x
x = np.array([3.0, 2.0])

# Number of diffusion steps
T = 40

# Number of independent particles.
# They all begin at the SAME x.
N = 500

# Noise schedule
# Made slightly larger than a typical DDPM so the effect is
# visually obvious with only 40 steps.
betas = np.linspace(0.01, 0.12, T)

alphas = 1.0 - betas

# cumulative alpha:
# alpha_bar[t] = alpha_1 * alpha_2 * ... * alpha_t
alpha_bars = np.cumprod(alphas)


# ---------------------------------------------------------
# 2. SIMULATE THE FORWARD MARKOV CHAIN
# ---------------------------------------------------------

# shape:
# positions[t, particle, coordinate]
positions = np.zeros((T + 1, N, 2))

# Every particle begins at x
positions[0, :, :] = x

for t in range(1, T + 1):

    beta_t = betas[t - 1]

    # Each particle gets an independent epsilon ~ N(0, I_2)
    epsilon = np.random.randn(N, 2)

    positions[t] = (
        np.sqrt(1 - beta_t) * positions[t - 1]
        + np.sqrt(beta_t) * epsilon
    )


# ---------------------------------------------------------
# 3. CHOOSE ONE PARTICLE TO FOLLOW
# ---------------------------------------------------------

highlight = 0

trajectory = positions[:, highlight, :]


# ---------------------------------------------------------
# 4. CREATE FIGURE
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 8))

ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)

ax.set_aspect("equal")

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)

ax.grid(alpha=0.2)

ax.set_xlabel("$z_1$")
ax.set_ylabel("$z_2$")


# Initial point x
ax.scatter(
    x[0],
    x[1],
    s=180,
    marker="*",
    label="$x$"
)


# Cloud of particles
cloud = ax.scatter(
    [],
    [],
    s=15,
    alpha=0.25,
    label="samples"
)


# Highlighted particle
current_point = ax.scatter(
    [],
    [],
    s=100,
    zorder=5,
    label="one trajectory"
)


# Path followed by highlighted particle
path_line, = ax.plot(
    [],
    [],
    linewidth=2
)


# Theoretical mean
mean_point = ax.scatter(
    [],
    [],
    marker="x",
    s=120,
    linewidths=3,
    label="theoretical mean"
)


# text
info_text = ax.text(
    0.02,
    0.98,
    "",
    transform=ax.transAxes,
    va="top",
    fontsize=11
)


ax.legend(loc="lower left")


# ---------------------------------------------------------
# 5. ANIMATION FUNCTION
# ---------------------------------------------------------

def update(t):

    # -----------------------------------------------------
    # Particle cloud
    # -----------------------------------------------------

    cloud.set_offsets(positions[t])


    # -----------------------------------------------------
    # Highlight one trajectory
    # -----------------------------------------------------

    z_current = trajectory[t]

    current_point.set_offsets([z_current])

    path_line.set_data(
        trajectory[:t+1, 0],
        trajectory[:t+1, 1]
    )


    # -----------------------------------------------------
    # Theoretical q(z_t | x)
    # -----------------------------------------------------

    if t == 0:

        mean = x
        variance = 0

    else:

        alpha_bar = alpha_bars[t - 1]

        mean = np.sqrt(alpha_bar) * x

        variance = 1 - alpha_bar


    mean_point.set_offsets([mean])


    # -----------------------------------------------------
    # Text
    # -----------------------------------------------------

    info_text.set_text(
        f"Diffusion step t = {t}\n"
        f"Mean = ({mean[0]:.2f}, {mean[1]:.2f})\n"
        f"Variance = {variance:.3f}"
    )


    ax.set_title(
        r"Forward diffusion: $q(z_t\mid x)$"
    )

    return (
        cloud,
        current_point,
        path_line,
        mean_point,
        info_text
    )


# ---------------------------------------------------------
# 6. BUILD ANIMATION
# ---------------------------------------------------------

animation = FuncAnimation(
    fig,
    update,
    frames=T + 1,
    interval=300,
    blit=False
)

plt.close(fig)

HTML(animation.to_jshtml())